# Smart Waste Classification - MobileNetV2 Training
Jalankan di Google Colab dengan GPU T4 (Runtime > Change runtime type > T4 GPU).
Output: `model.pth`, `confusion_matrix.png`, `classification_report.json`, `training_curves.png`.

In [ ]:
# 1. Download Kaggle dataset
import os
!pip install -q kaggle scikit-learn seaborn
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ 2>/dev/null && chmod 600 ~/.kaggle/kaggle.json || true
!kaggle datasets download -d mostafaabla/garbage-classification --unzip -p /content/raw

In [ ]:
# 2. Prepare dataset: 400 gambar/kelas, 3 kelas, split 70/15/15
import random, shutil
from pathlib import Path

SEED, PER_CLASS = 42, 400
MAP = {'organic': ['biological'], 'plastic': ['plastic'], 'paper': ['paper', 'cardboard']}
OUT = Path('/content/data')
# cari folder yang berisi kelas 'biological' (nama folder induk bisa beda: garbage_classification, Garbage classification, dst)
hits = [p.parent for p in Path('/content/raw').rglob('biological') if p.is_dir()]
assert hits, f"folder 'biological' tidak ditemukan. Isi /content/raw: {[str(p) for p in Path('/content/raw').rglob('*') if p.is_dir()][:20]}"
RAW = hits[0]; print('Dataset di:', RAW, '| kelas:', sorted(p.name for p in RAW.iterdir() if p.is_dir()))

random.seed(SEED)
for cls, srcs in MAP.items():
    files = sorted(f for s in srcs for f in (RAW / s).glob('*') if f.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    assert len(files) >= PER_CLASS, f'{cls}: {len(files)} < {PER_CLASS}'
    files = random.sample(files, PER_CLASS)
    n_tr, n_va = int(PER_CLASS * .70), int(PER_CLASS * .15)
    parts = {'train': files[:n_tr], 'val': files[n_tr:n_tr + n_va], 'test': files[n_tr + n_va:]}
    for split, fs in parts.items():
        d = OUT / split / cls; d.mkdir(parents=True, exist_ok=True)
        for i, f in enumerate(fs): shutil.copy(f, d / f'{cls}_{i:03d}{f.suffix.lower()}')

for s in ('train', 'val', 'test'): print(s, {c: len(list((OUT / s / c).glob('*'))) for c in MAP})

In [ ]:
# 3. Train MobileNetV2 + Export Evaluation
import json, time
import torch, torch.nn as nn
import matplotlib.pyplot as plt, seaborn as sns
from torchvision import datasets, models, transforms as T
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix

torch.manual_seed(42)
RES = Path('/content/results'); RES.mkdir(exist_ok=True)
MODEL = Path('/content/model.pth')
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_tr = T.Compose([T.Resize((224, 224)), T.RandomHorizontalFlip(), T.RandomRotation(15), T.ColorJitter(.2, .2), T.ToTensor(), norm])
tf_ev = T.Compose([T.Resize((224, 224)), T.ToTensor(), norm])

ds = {s: datasets.ImageFolder(OUT / s, tf_tr if s == 'train' else tf_ev) for s in ('train', 'val', 'test')}
dl = {s: DataLoader(d, batch_size=32, shuffle=(s == 'train'), num_workers=2) for s, d in ds.items()}
classes = ds['train'].classes

m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
m.classifier = nn.Sequential(nn.Dropout(.3), nn.Linear(m.last_channel, 128), nn.ReLU(), nn.Dropout(.2), nn.Linear(128, len(classes)))
m.to(dev)
opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

def step(split, train=False):
    m.train(train); tot = correct = n = 0
    with torch.set_grad_enabled(train):
        for x, y in dl[split]:
            x, y = x.to(dev), y.to(dev)
            out = m(x); loss = loss_fn(out, y)
            if train: opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return tot / n, correct / n

hist = {'tl': [], 'vl': [], 'ta': [], 'va': []}
best, bad = 1e9, 0
for ep in range(20):
    tl, ta = step('train', True); vl, va = step('val')
    for k, v in zip(hist, (tl, vl, ta, va)): hist[k].append(v)
    print(f'ep {ep+1:02d} | loss {tl:.3f}/{vl:.3f} | acc {ta:.3f}/{va:.3f}')
    if vl < best: best, bad = vl, 0; torch.save({'state': m.state_dict(), 'classes': classes}, MODEL)
    else:
        bad += 1
        if bad >= 4: print('Early stopping'); break

m.load_state_dict(torch.load(MODEL)['state']); m.eval()
ys, ps = [], []
with torch.no_grad():
    for x, y in dl['test']: ps += m(x.to(dev)).argmax(1).cpu().tolist(); ys += y.tolist()
acc = sum(a == b for a, b in zip(ys, ps)) / len(ys)
print(f'=== TEST ACCURACY: {acc * 100:.2f}% ===')
rep = classification_report(ys, ps, target_names=classes, output_dict=True)
(RES / 'classification_report.json').write_text(json.dumps(rep, indent=2))
cm = confusion_matrix(ys, ps)
plt.figure(figsize=(5, 4)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Prediksi'); plt.ylabel('Aktual'); plt.tight_layout(); plt.savefig(RES / 'confusion_matrix.png', dpi=150)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(hist['tl'], label='train'); ax[0].plot(hist['vl'], label='val'); ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(hist['ta'], label='train'); ax[1].plot(hist['va'], label='val'); ax[1].set_title('Accuracy'); ax[1].legend()
plt.tight_layout(); plt.savefig(RES / 'training_curves.png', dpi=150)

# Zip artifacts
!zip -q -j /content/smart_waste_ml_artifacts.zip /content/model.pth /content/results/*
print('Download /content/smart_waste_ml_artifacts.zip dan ekstrak ke repo ml/')